# 01 — BCCD source inspection

**Purpose:** Audit real microscopy files, VOC boxes, source classes and split-file contents.

**Required inputs:** Configured BCCD root with `JPEGImages/`, `Annotations/`, `ImageSets/Main/` (direct or nested `BCCD/`).

**Saved outputs:** `outputs/dataset_inspection.json`, original gallery and class chart.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## 2. Inspect source files

Image sizes come from image pixels, not XML paths or stale XML dimensions. A standard release is near 364 images; variations are allowed.

In [ ]:
from src.bccd import inspect_dataset, locate_bccd
from IPython.display import display
import pandas as pd
source_root = locate_bccd(cfg["dataset_root"])
print("Source:", source_root)
print("ImageSets/Main exists:", (source_root / "ImageSets" / "Main").is_dir())
inspection = inspect_dataset(cfg, strict=False)
display(pd.DataFrame([{"measure": k, "value": inspection[k]} for k in
    ("total_images", "total_xml", "total_wbc_boxes", "images_with_wbc", "images_without_wbc",
     "missing_xml", "unreadable_images", "invalid_boxes_or_xml")]))
display(pd.DataFrame(inspection["class_distribution"].items(), columns=["class", "boxes"]))
display(pd.DataFrame([{"file": k, "lines": len(v), "first_entries": v[:5]} for k,v in inspection["source_split_files"].items()]))

## 3. View real images and XML

VOC boxes are color coded: RBC red, WBC green, platelets blue. XML examples are read from the configured dataset.

In [ ]:
from src.visualization import source_gallery, class_chart
if inspection["records"]:
    display(source_gallery(inspection["records"], out / "figures" / "original_bccd_examples.png", count=9))
    display(class_chart(inspection["class_distribution"], out / "figures" / "source_class_counts.png"))
    xml_example = Path(inspection["records"][0]["xml"])
    print("XML example:", xml_example.name)
    print(xml_example.read_text(encoding="utf-8")[:1600])
    display(pd.DataFrame([{"image": r["id"], "pixels": f"{r['width']}x{r['height']}",
                           "xml_size": r["xml_size"], "clipped_boxes": len(r["clipping"])}
                          for r in inspection["records"][:10]]))

## 4. Integrity gate

Stop here if an image, annotation or bounding box is missing or unreadable.

In [ ]:
assert inspection["ready"], "FAIL — source integrity errors; inspect outputs/dataset_inspection.json"
print("PASS — source inspection complete. Next: notebook 02.")

## Next notebook

Open **02_BCCD_Preprocessing.ipynb**.